In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import re
import os
import glob
import sys
from tqdm import tqdm
import datetime
import gc
import numpy as np
import tensorflow as tf

In [3]:
project_path = 'drive/MyDrive/handwriting_recognition'

In [4]:
sys.path.insert(0, project_path)

In [5]:
sys.path

['drive/MyDrive/handwriting_recognition',
 '/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython']

In [6]:
!pip install "zarr>=2.16.0,<3.0.0" numcodecs

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 123.2 MB/s eta 0:00:00
  Created wheel for asciitree: filename=asciitree-0.3.3-py3-none-any.whl size=5031 sha256=acaded43f4a3256f7591f60dc2fc22b978abe748fda72652194e774e305ffa78
  Stored in directory: /root/.cache/pip/wheels/a5/d7/98/f56ae733748cd0fa577172bda0e73e0b1f1793c98e09b9e458
Successfully built asciitree


In [7]:
from src.io.exceptions import DeserializationError
from src.io.logging import LoggerFactory
from src import SelectiveDataLoader
from src.io import load

In [8]:
dataset_path = f"{project_path}/data/datasets/complete_dataset_20260122-142244.tar.gz"

In [9]:
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [10]:
logger = LoggerFactory.get_notebook_logger("code_1", log_dir=f"./logs_{current_time}")

In [11]:
!tar -xzf {dataset_path} -C ./

In [12]:
loader = SelectiveDataLoader(
        dataset_path="./complete_dataset_20260122-142244.zarr",
        load_fn=load
        )

12:18:57 - ZarrSerializer - INFO - Initialized with blosc compression (level 5)
12:18:57 - ZarrSerializer - INFO - ======================================================================
12:18:57 - ZarrSerializer - INFO - Loading Zarr dataset: complete_dataset_20260122-142244.zarr
12:18:57 - ZarrSerializer - INFO - ======================================================================
12:18:57 - ZarrSerializer - INFO - 📂 Loading partition (lazy mode)...
12:18:57 - ZarrSerializer - INFO -    ├─ train: 1,268,876 patches (~34851MB, lazy-loaded)
12:18:57 - ZarrSerializer - INFO -    ├─ validation: 206,216 patches (~5664MB, lazy-loaded)
12:18:57 - ZarrSerializer - INFO -    ├─ test: 51,067 patches (~1403MB, lazy-loaded)
12:18:57 - ZarrSerializer - INFO -    └─ Partition loaded (data not yet in memory)
12:18:57 - ZarrSerializer - INFO - 📂 Loading 'labels': (1526159,), 5.8MB
12:18:57 - ZarrSerializer - INFO - 📂 Loading metadata: 5 items
12:18:57 - ZarrSerializer - INFO - ======================

In [13]:
from src.models import CNNBackbone, CNNTransformerBackbone, build_siamese_model, build_embedding_model

In [14]:
backbone = CNNBackbone()

In [15]:
model = build_embedding_model(
    backbone=backbone,
    input_shape=(60, 53, 1),
    embedding_dim=256,
    name="cnn_embedding_model"
)

In [16]:
model.summary()

Model: "cnn_embedding_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60, 53, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 60, 53, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 53, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 60, 53, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 30, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 30, 26, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30, 26, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 30, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 15, 13, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 15, 13, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 15, 13, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 15, 13, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 7, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 7, 6, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 7, 6, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 7, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_projection (Dense)    │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ l2_normalize (Lambda)           │ (None, 256)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 455,552 (1.74 MB)

 Trainable params: 454,592 (1.73 MB)

 Non-trainable params: 960 (3.75 KB)

In [17]:
subset = loader.load_dataset_by_authors(range(100), show_progress_bar=True)

07:40:30 - ZarrSerializer - INFO - ======================================================================
07:40:30 - ZarrSerializer - INFO - Loading Zarr dataset: complete_dataset_20260122-142244.zarr
07:40:30 - ZarrSerializer - INFO - ======================================================================
07:40:30 - ZarrSerializer - INFO - 📂 Loading partition (lazy mode)...
07:40:30 - ZarrSerializer - INFO -    ├─ train: 1,268,876 patches (~34851MB, lazy-loaded)
07:40:30 - ZarrSerializer - INFO -    ├─ validation: 206,216 patches (~5664MB, lazy-loaded)
07:40:30 - ZarrSerializer - INFO -    ├─ test: 51,067 patches (~1403MB, lazy-loaded)
07:40:30 - ZarrSerializer - INFO -    └─ Partition loaded (data not yet in memory)
07:40:30 - ZarrSerializer - INFO - 📂 Loading 'labels': (1526159,), 5.8MB
07:40:30 - ZarrSerializer - INFO - 📂 Loading metadata: 5 items
07:40:30 - ZarrSerializer - INFO - ======================================================================
07:40:30 - ZarrSerializer - INF

Building subset dataset: 100%|██████████| 300/300 [00:39<00:00,  7.53it/s]


In [18]:
subset.keys()

dict_keys(['train', 'validation', 'test', 'num_authors', 'author_names', 'patch_shape', 'metadata', 'global_to_local', 'local_to_global'])

In [26]:
del subset
gc.collect()

104

In [17]:
def get_subset(loader: SelectiveDataLoader, authors_range: list[int] | range, normalize: bool = True, add_channel_dim: bool = True, show_progress_bar: bool = True) -> dict:
  subset = loader.load_dataset_by_authors(
      authors_id=authors_range,
      show_progress_bar=show_progress_bar
  )

  result = {}

  for split in ['train', 'validation', 'test']:
    X, y = subset[split]

    if normalize:
      X = X.astype('float32') / 255.0

    if add_channel_dim and X.ndim == 3:
      X = X[..., np.newaxis]

    result[split] = (X, y)

  del subset
  gc.collect()

  return result


In [18]:
data = get_subset(
    loader=loader,
    authors_range=range(50),
    normalize=True,
    add_channel_dim=True
)

12:19:00 - ZarrSerializer - INFO - ======================================================================
12:19:00 - ZarrSerializer - INFO - Loading Zarr dataset: complete_dataset_20260122-142244.zarr
12:19:00 - ZarrSerializer - INFO - ======================================================================
12:19:00 - ZarrSerializer - INFO - 📂 Loading partition (lazy mode)...
12:19:00 - ZarrSerializer - INFO -    ├─ train: 1,268,876 patches (~34851MB, lazy-loaded)
12:19:00 - ZarrSerializer - INFO -    ├─ validation: 206,216 patches (~5664MB, lazy-loaded)
12:19:00 - ZarrSerializer - INFO -    ├─ test: 51,067 patches (~1403MB, lazy-loaded)
12:19:00 - ZarrSerializer - INFO -    └─ Partition loaded (data not yet in memory)
12:19:00 - ZarrSerializer - INFO - 📂 Loading 'labels': (1526159,), 5.8MB
12:19:00 - ZarrSerializer - INFO - 📂 Loading metadata: 5 items
12:19:00 - ZarrSerializer - INFO - ======================================================================
12:19:00 - ZarrSerializer - INF

Building subset dataset: 100%|██████████| 150/150 [00:23<00:00,  6.41it/s]


In [19]:
data.keys()

dict_keys(['train', 'validation', 'test'])

In [20]:
len(data['train']), len(data['validation']), len(data['test'])

(2, 2, 2)

In [21]:
len(data['train'][0]), len(data['train'][1])

(153701, 153701)

In [22]:
len(data['validation'][0]), len(data['validation'][1])

(24954, 24954)

In [23]:
len(data['test'][0]), len(data['test'][1])

(6201, 6201)

In [27]:
del data
gc.collect()

0

In [19]:
from src.datasets import DataGenerator

In [20]:
train_gen = DataGenerator(
    data_partion=data['train'],
    batch_size=64,
    shuffle=True,
    use_augmentation=True,
    seed=42
)

In [24]:
import time

def benchmark_generator(generator: DataGenerator, n_epochs: int = 3, n_batches: int | None = None) -> None:
  times = []

  for epoch in range(n_epochs):
    start = time.time()

    for i, batch in enumerate(generator):
      if n_batches is not None and i >= n_batches:
        break

    epoch_time = time.time() - start
    times.append(epoch_time)

    logger.info(f"Epoch {epoch + 1}: {epoch_time:.2f}s")

  logger.info(f"\n{'='*50}")
  logger.info(f"Average time per epoch: {np.mean(times):.2f}s ± {np.std(times):.2f}s")
  logger.info(f"Best time per epoch: {np.min(times):.2f}s")
  logger.info(f"Worst time per epoch: {np.max(times):.2f}s")

  if n_batches:
    total_batches = len(generator)
    estimated_full = np.mean(times) * (total_batches / n_batches)
    logger.info(f"Estimated time to finish: {estimated_full:.2f}s")

In [26]:
benchmark_generator(train_gen, n_epochs=1, n_batches=100)

INFO - Epoch 1: 3.79s
INFO - 
INFO - Average time per epoch: 3.79s ± 0.00s
INFO - Best time per epoch: 3.79s
INFO - Worst time per epoch: 3.79s
INFO - Estimated time to finish: 91.11s


In [28]:
benchmark_generator(train_gen, n_epochs=3, n_batches=64)

INFO - Epoch 1: 2.46s
INFO - Epoch 2: 2.70s
INFO - Epoch 3: 2.64s
INFO - 
INFO - Average time per epoch: 2.60s ± 0.10s
INFO - Best time per epoch: 2.46s
INFO - Worst time per epoch: 2.70s
INFO - Estimated time to finish: 97.62s


In [29]:
benchmark_generator(train_gen, n_epochs=10, n_batches=64)

INFO - Epoch 1: 2.43s
INFO - Epoch 2: 2.50s
INFO - Epoch 3: 2.52s
INFO - Epoch 4: 2.56s
INFO - Epoch 5: 2.78s
INFO - Epoch 6: 2.60s
INFO - Epoch 7: 2.55s
INFO - Epoch 8: 2.55s
INFO - Epoch 9: 2.53s
INFO - Epoch 10: 2.82s
INFO - 
INFO - Average time per epoch: 2.59s ± 0.11s
INFO - Best time per epoch: 2.43s
INFO - Worst time per epoch: 2.82s
INFO - Estimated time to finish: 97.03s


In [30]:
val_gen = DataGenerator(
    data_partion=data['validation'],
    batch_size=32,
    use_augmentation=False,
    shuffle=False,
)

In [31]:
len(train_gen), len(val_gen)

(2402, 780)

In [32]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [33]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    verbose=1
)

Epoch 1/10
2402/2402 ━━━━━━━━━━━━━━━━━━━━ 126s 47ms/step - accuracy: 0.0220 - loss: 7.6827 - val_accuracy: 0.0303 - val_loss: 6.9869
Epoch 2/10
2402/2402 ━━━━━━━━━━━━━━━━━━━━ 100s 42ms/step - accuracy: 0.0300 - loss: 6.6298 - val_accuracy: 0.0267 - val_loss: 6.7566
Epoch 3/10
2402/2402 ━━━━━━━━━━━━━━━━━━━━ 100s 42ms/step - accuracy: 0.0272 - loss: 6.4461 - val_accuracy: 0.0278 - val_loss: 6.2354
Epoch 4/10
2402/2402 ━━━━━━━━━━━━━━━━━━━━ 102s 42ms/step - accuracy: 0.0296 - loss: 6.2004 - val_accuracy: 0.0652 - val_loss: 5.5798
Epoch 5/10
2402/2402 ━━━━━━━━━━━━━━━━━━━━ 102s 42ms/step - accuracy: 0.0365 - loss: 5.8627 - val_accuracy: 0.0435 - val_loss: 5.6398
Epoch 6/10
2402/2402 ━━━━━━━━━━━━━━━━━━━━ 100s 42ms/step - accuracy: 0.0525 - loss: 5.6831 - val_accuracy: 0.0753 - val_loss: 5.4676
Epoch 7/10
2402/2402 ━━━━━━━━━━━━━━━━━━━━ 102s 42ms/step - accuracy: 0.0589 - loss: 5.4720 - val_accuracy: 0.0929 - val_loss: 5.2319
Epoch 8/10
2402/2402 ━━━━━━━━━━━━━━━━━━━━ 101s 42ms/step - accuracy: 

In [34]:
test_gen = DataGenerator(
    data_partion=data['test'],
    batch_size=32,
    use_augmentation=False,
    shuffle=False,
)

In [35]:
test_loss, test_acc = model.evaluate(test_gen)

194/194 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.2622 - loss: 3.9829


In [36]:
logger.info(f"Test accuracy: {test_acc:.4f}")

INFO - Test accuracy: 0.1671


In [28]:
from tensorflow.keras import layers, models, ops

In [29]:
def build_siamese_model_two_inputs(
        embedding_model: tf.keras.Model,
        input_shape: tuple[int, int, int] = (180, 160, 1),
) -> tf.keras.Model:
    """
    Construye modelo Siamese que recibe DOS inputs separados.

    Compatible con PairGenerator que retorna [X1, X2], y

    Inputs:
        - image1: (batch, H, W, C)
        - image2: (batch, H, W, C)
    Output: (batch, 2, embedding_dim)
    """
    input1 = layers.Input(shape=input_shape, name="image1")
    input2 = layers.Input(shape=input_shape, name="image2")

    embedding1 = embedding_model(input1)  # (batch, embedding_dim)
    embedding2 = embedding_model(input2)

    embeddings = ops.stack([embedding1, embedding2], axis=1)  # (batch, 2, embedding_dim)

    return models.Model(
        inputs=[input1, input2],
        outputs=embeddings,
        name="siamese_model_two_inputs"
    )

In [26]:
backbone = CNNTransformerBackbone(num_layers=4, num_heads=8, ff_dim=1024)
embedding_model = build_embedding_model(
    backbone=backbone,
    input_shape=(180, 160, 1),
    embedding_dim=256
)

In [30]:
siamese_model = build_siamese_model_two_inputs(
    embedding_model=embedding_model,
    input_shape=(180, 160, 1)
)

In [31]:
siamese_model.summary()

Model: "siamese_model_two_inputs"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image1 (InputLayer) │ (None, 180, 160,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image2 (InputLayer) │ (None, 180, 160,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_model     │ (None, 256)       │ 10,976,128 │ image1[0][0],     │
│ (Functional)        │                   │            │ image2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 2, 256)    │          0 │ embedding_model[… │
│                     │                   │            │ embedding_model[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,976,128 (41.87 MB)

 Trainable params: 10,975,168 (41.87 MB)

 Non-trainable params: 960 (3.75 KB)

In [32]:
siamese_model.compile(
    optimizer='adam',
    loss='contrastive_loss'
)

In [33]:
from src.datasets import PairGenerator

In [35]:
train_gen = PairGenerator(data['train'], batch_size=32)
val_gen = PairGenerator(data['validation'], batch_size=32)

In [37]:
history = siamese_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

TypeError: `output_signature` must contain objects that are subclass of `tf.TypeSpec` but found <class 'list'> which is not.